**Drug pridiction by using softmax function**

In [1]:
# Import important libaries

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder , StandardScaler , OneHotEncoder

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Softmax

In [2]:
# Read data from local file
df = pd.read_csv(r"C:\Users\Lenovo\Downloads\drug200.csv")
df

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY
...,...,...,...,...,...,...
195,56,F,LOW,HIGH,11.567,drugC
196,16,M,LOW,HIGH,12.006,drugC
197,52,M,NORMAL,HIGH,9.894,drugX
198,23,M,NORMAL,NORMAL,14.020,drugX


In [3]:
# Check null values
df.isnull().sum()

Age            0
Sex            0
BP             0
Cholesterol    0
Na_to_K        0
Drug           0
dtype: int64

In [4]:
df.dtypes

Age              int64
Sex             object
BP              object
Cholesterol     object
Na_to_K        float64
Drug            object
dtype: object

In [5]:
# Features & Target
X = df.drop(columns=["Drug"])
y = df["Drug"]

In [6]:
from sklearn.preprocessing import LabelEncoder

# Encode target column
le = LabelEncoder()
y = le.fit_transform(y)

In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

cat_cols = ['Sex', 'BP', 'Cholesterol']
num_cols = ['Age', 'Na_to_K']

pre = ColumnTransformer([
    ('cat', OneHotEncoder(), cat_cols),
    ('num', StandardScaler(), num_cols)
])

In [8]:
# Split into Training and Testing
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
# Fit + Transform
X_train_pre = pre.fit_transform(X_train)
X_test_pre = pre.transform(X_test)

In [10]:
# Make a model using deep learning
model = Sequential()

model.add(Dense(32, input_dim=X_train_pre.shape[1], activation='relu'))
model.add(Dense(16, activation='relu'))
model.add(Dense(5, activation='softmax')) 

C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [11]:
# Compile and fitted  into  splited data
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(X_train_pre, y_train, epochs=40, batch_size=16, validation_split=0.2)

Epoch 1/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - accuracy: 0.1953 - loss: 1.6661 - val_accuracy: 0.3750 - val_loss: 1.5266
Epoch 2/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.2969 - loss: 1.5715 - val_accuracy: 0.4375 - val_loss: 1.4523
Epoch 3/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.4375 - loss: 1.4975 - val_accuracy: 0.5312 - val_loss: 1.3884
Epoch 4/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.4531 - loss: 1.4352 - val_accuracy: 0.5312 - val_loss: 1.3356
Epoch 5/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4609 - loss: 1.3823 - val_accuracy: 0.5312 - val_loss: 1.2890
Epoch 6/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4609 - loss: 1.3352 - val_accuracy: 0.5312 - val_loss: 1.2486
Epoch 7/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4609 - loss: 1.2936 - val_accuracy: 0.5312 - val_loss: 1.2121
Epoch 8/40
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.4609 - loss: 1.2544 - val_accuracy: 0.5312 - val_loss: 1.1745
E

In [12]:
# Check accuarcy
loss, acc = model.evaluate(X_test_pre, y_test)
print("Accuracy:", acc)


2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.8500 - loss: 0.4523 
Accuracy: 0.8500000238418579


In [13]:
df.head(20)

,Age,Sex,BP,Cholesterol,Na_to_K,Drug
0,23,F,HIGH,HIGH,25.355,drugY
1,47,M,LOW,HIGH,13.093,drugC
2,47,M,LOW,HIGH,10.114,drugC
3,28,F,NORMAL,HIGH,7.798,drugX
4,61,F,LOW,HIGH,18.043,drugY
5,22,F,NORMAL,HIGH,8.607,drugX
6,49,F,NORMAL,HIGH,16.275,drugY
7,41,M,LOW,HIGH,11.037,drugC
8,60,M,NORMAL,HIGH,15.171,drugY
9,43,M,LOW,NORMAL,19.368,drugY


In [14]:
# Take live data 
live = pd.DataFrame({
    'Age': [43],
    'Sex': ['F'],
    'BP': ['HIGH'],
    'Cholesterol': ['HIGH'],
    'Na_to_K': [13.093]
})


In [15]:
live_pre = pre.transform(live)

# Predict
pred = model.predict(live_pre)

# Convert to class
pred_class = np.argmax(pred)

# Decode
drug_name = le.inverse_transform([pred_class])

print("Predicted Drug:", drug_name[0])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 79ms/step
Predicted Drug: drugA
